# Ihminen-silmukassa: Ennakkoportit, riskiluokittelu ja tarkastuskirjanpito

Tämän oppitunnin README esittelee Ihminen-silmukassa -mallin lyhyellä pätkällä, jossa käyttäjältä kysytään `HYVÄKSY` tai `HYLKÄÄ` sen jälkeen kun agentti on jo tuottanut vastauksen. Tämä malli on hyvä lähtökohta, mutta tuotantotason HITL-toteutuksissa tarvitaan usein kolme lisäosaa:

1. **Ennakkoportti**, joka suoritetaan **ennen** kuin agentti tekee riskialttiin toimenpiteen, jotta kustannukset, peruuttamattomuus ja viive pysyvät hallinnassa.
2. **Riskiluokittelu**, jossa matalariskiset toimet suoritetaan automaattisesti, keskiriskiset toimet hyväksytään erissä ja vain korkeimman riskin toimet vaativat ihmisen päätöksen.
3. **Tarkastuskirjanpito ja uudelleenkäsittely-silmukka**, jossa jokainen portin päätös tallennetaan JSONL-muotoon, ja hylkäys antaa rakenteellisen syyn agentille uudelleenkäsittelyä varten sen sijaan, että vain tulostettaisiin `Uudelleentarkistetaan...`.

Tämä muistikirja rakentaa jokaisen näistä samoille primitiiveille kuin `06-system-message-framework.ipynb`. Se suoritetaan kokonaisuudessaan `DEMO_MODE = True` -tilassa (ei tarvita interaktiivista syötettä) tai oikeiden `input()`-kyselyjen kanssa, kun `DEMO_MODE = False`. Huom: DEMO_MODE-tilassa kolmannen tavoitteen uudelleensyötön toisto on käsikirjoitettu, jotta silmukan mekanismi näkyy kokonaisuudessaan. Todellinen uudelleentarkistukseen perustuva uudelleenluokittelu vaatii `DEMO_MODE = False` ja operaattorin.

**Ei kuulu käsiteltäviin aiheisiin (käsitellään muissa oppitunneissa):** todennus ja käyttöoikeuksien hallinta (Oppitunti 06 README uhka 2), työkalukutsun välikerros (Oppitunti 14 MAF syväluotaus), moniagenttikeskustelumallit.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Kuvio 1: Ennakko-toiminnon portti

README-tiedoston HITL-pätkä kutsuu agenttia ensin ja pyytää sitten käyttäjältä hyväksynnän tulokselle. Tämä on **jälkitoiminto**-kulku. Agentti on jo suorittanut toiminnon, joten LLM-kutsun kustannus on jo maksettu, ja kaikki sivuvaikutukset (lähetetty sähköposti, kirjoitettu tietokantarivi, julkaistu kommentti) on jo tapahtunut.

**Ennakko-toiminto**-kulku lisää portin ennen kuin agentti suorittaa riskialttiin vaiheen. Agentti ehdottaa toimintoa, portti päättää, suorutetaanko, ja sivuvaikutus tapahtuu vain hyväksynnän jälkeen.

| Näkökulma | Jälkitoiminnon hyväksyntä (README-pätkä) | Ennakko-toiminnon portti (tämä muistikirja) |
|---|---|---|
| Milloin hyväksyntä suoritetaan? | Sen jälkeen kun agentti on tuottanut tuloksen | Ennen kuin mikään sivuvaikutus suoritetaan |
| LLM-kustannus hylkäyksessä | Jo maksettu | Maksetaan vain ehdotuksesta, ei toiminnosta |
| Peruuttamattomat sivuvaikutukset | Mahdollisia (toiminto on jo tapahtunut) | Estetty |
| Tarkastettavuus | Hyväksyntä on tulostuslause | Hyväksyntä on JSON-tietue aikaleimalla, toiminnolla, syyllä |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Kuvio 2: Riskiluokitus

Kaikki toimet eivät vaadi ihmisen hyväksyntää. Julkisen rajapinnan lukuoikeudellinen haku on eri asia kuin asiakkaalle sähköpostin lähettäminen. Näiden käsitteleminen samalla tavalla hukkaa toimijan huomion ja hidastaa agenttia.

Yksinkertainen 3-tasoinen malli:

| Taso | Esimerkkejä | Hyväksyntäprosessi |
|---|---|---|
| `matala` (vain luku) | Tietopohjan haku, lentovaihtoehtojen etsiminen, julkisen web-sivun hakeminen | Automaattinen suoritus, kirjattu tarkastusta varten |
| `keskitaso` (halpa muutos) | Tuloksen välimuistitus, laskurin kasvatus, muistutuksen ajastus | Automaattinen suoritus, mutta ryhmitelty päivittäiseen tarkastukseen |
| `korkea` (ulkopuolinen tai peruuttamaton) | Sähköpostin lähetys, kortilta veloittaminen, julkiselle kanavalle postaus | Estetään ilman ihmisen hyväksyntää |

Tämä on yksi luokitus. Tuotantojärjestelmissä käytetään usein tarkempia luokkia (esim. AWS IAM -käyttöoikeustasot, roolipohjaiset käyttöoikeusluokat). Alla oleva 3-tasoinen versio on pienin hyödyllinen versio agentille, joka yhdistää lukuoikeuden ja sivuvaikutteiset toimet.

Alla oleva luokittelija käyttää avainsanaheuristiikkaa, jotta demo pysyy määrääntyvänä ja edullisena. Tuotantojärjestelmässä tämä vaihdettaisiin opetettuun luokittelijaan tai politiikkamoottoriin.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Kuvio 3: Auditointiloki ja revisiosilmukka

`print("Vastaus hyväksytty.")` ei ole auditointiloki. Luottamuksen vuoksi jokainen porttipäätös tulisi tallentaa strukturoituna tapahtumana, jota voit myöhemmin kysellä, toistaa tai liittää tapauksen tarkasteluun.

Kaksi osaa:

1. **Pelkästään lisättävä JSONL.** Yksi rivi päätöstä kohden, sisältäen aikaleiman, toiminnon, tason, päätöksen, syyn. Helppo grepata, helppo lähettää myöhemmin oikeaan lokivarastoon.
2. **Revisiosilmukka hylkäyksessä.** Kun portti palauttaa `deny`, agentti kehottaa itseään uudelleen hylkäyksen syyn kanssa kontekstissa, jotta seuraava ehdotus voi välttää ongelman.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Lisäresurssit

Useat muut julkiset projektit toteuttavat näitä HITL-malleja eri variaatioina. Vertaa lähestymistapoja löytääksesi oman pinoosi sopivan:

- **LangChain** ihmisen silmukassa -työkalukääreet ([dokumentaatio](https://python.langchain.com/docs/integrations/tools/human_tools)): työkalukääreet, jotka pysäyttävät suorituksen ihmisen syötteen odottamiseksi.
- **AutoGen** `UserProxyAgent` ([v0.2 dokumentaatio](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ on uudelleenrakentanut tämän): käyttää agenttiroolia edustamaan ihmistä moni-agenttikeskusteluissa.
- **Microsoft Agent Framework (MAF)** funktion kutsujen välikerros ([dokumentaatio](https://learn.microsoft.com/agent-framework/)): välikerros, joka toimii kaikkien työkalujen/funktioiden kutsujen ympärillä, sopii ehdolliseen logiikkaan ja hyväksyntäprosesseihin.

Jokainen projekti käsittelee kolmea alakuvioita eri tavalla: LangChain käärii ne työkaluiksi, AutoGen käyttää agenttiroolia ja Microsoft Agent Framework käyttää funktionkutsujen välikerrosta. Lue yksi tai kaksi toteutusta kokonaisuudessaan ennen oman agenttisi suunnittelun valintaa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
